In [3]:
import pandas as pd
df = pd.read_csv(r"E:\Research_Paper\oil_gas_ai\data\features\extraction_training_dataset.csv")
print(df['extraction_rate_bpd'].describe())
print(df['extraction_rate_bpd'].nunique(), 'unique values')

count    2.600000e+06
mean     2.285126e+04
std      2.545348e+04
min     -5.322217e+03
25%      9.513373e+01
50%      9.137192e+03
75%      3.940225e+04
max      8.121754e+04
Name: extraction_rate_bpd, dtype: float64
2372009 unique values


In [5]:
print(df[['pressure_gas_lift','extraction_rate_bpd']].corr())


                     pressure_gas_lift  extraction_rate_bpd
pressure_gas_lift                  NaN                  NaN
extraction_rate_bpd                NaN                  1.0


In [6]:
import pandas as pd

df = pd.read_csv(r"E:\Research_Paper\oil_gas_ai\data\features\extraction_training_dataset.csv")

print("=== SHAPE ===")
print(df.shape)

print("\n=== COLUMNS IN CSV ===")
print(df.columns.tolist())

print("\n=== NULL COUNT PER COLUMN ===")
print(df.isnull().sum())

print("\n=== FIRST 3 ROWS ===")
print(df.head(3))

=== SHAPE ===
(2600000, 12)

=== COLUMNS IN CSV ===
['pressure_gas_lift', 'pressure_production_casing', 'pressure_toptubing', 'temp_toptubing', 'temp_downholegauge', 'well_age_years', 'attribute_score', 'aggregate_quality_score', 'mtbf_days', 'mttr_days', 'availability_pct', 'extraction_rate_bpd']

=== NULL COUNT PER COLUMN ===
pressure_gas_lift             0
pressure_production_casing    0
pressure_toptubing            0
temp_toptubing                0
temp_downholegauge            0
well_age_years                0
attribute_score               0
aggregate_quality_score       0
mtbf_days                     0
mttr_days                     0
availability_pct              0
extraction_rate_bpd           0
dtype: int64

=== FIRST 3 ROWS ===
   pressure_gas_lift  pressure_production_casing  pressure_toptubing  \
0      -7.105427e-15                         0.0           -0.701410   
1      -7.105427e-15                         0.0           -0.701410   
2      -7.105427e-15               

In [7]:
import pandas as pd
import joblib
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

from pipelines.feature_contract_extraction import (
    FEATURE_COLUMNS,
    TARGET_COLUMN
)

# ─────────────────────────────────────────────
# 1. Load raw dataset (NO scaling)
# ─────────────────────────────────────────────
print("📂 Loading dataset...")
df = pd.read_csv("data/features/extraction_training_dataset.csv")
print(f"   Shape: {df.shape}")

# ─────────────────────────────────────────────
# 2. Check if data is pre-scaled and undo it
#    (values between -5 and 5 = already scaled)
# ─────────────────────────────────────────────
sample_val = df["pressure_gas_lift"].abs().max()

if sample_val < 100:
    print("\n⚠️  WARNING: Data appears to be pre-scaled (Z-score normalized).")
    print("   Re-generating raw features from build script is recommended.")
    print("   Proceeding with scaled data — test inputs must also be scaled.\n")
else:
    print("   ✅ Data is in raw units — no scaling needed.\n")

# ─────────────────────────────────────────────
# 3. Features and target — raw, no scaling
# ─────────────────────────────────────────────
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

print(f"   Feature columns : {FEATURE_COLUMNS}")
print(f"   Target range    : {y.min():.1f} to {y.max():.1f} BPD")
print(f"   Target mean     : {y.mean():.1f} BPD\n")

# ─────────────────────────────────────────────
# 4. Train-test split
# ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"   Train size: {len(X_train):,} | Test size: {len(X_test):,}")

# ─────────────────────────────────────────────
# 5. Model — XGBoost (NO scaling needed for trees)
# ─────────────────────────────────────────────
model = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,          # use all CPU cores
    tree_method="hist"  # faster training
)

print("\n🚀 Training model...")
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# ─────────────────────────────────────────────
# 6. Evaluate
# ─────────────────────────────────────────────
y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)  # true RMSE
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("\n" + "=" * 45)
print("   EXTRACTION MODEL PERFORMANCE")
print("=" * 45)
print(f"   RMSE : {rmse:>12,.3f} BPD")
print(f"   MAE  : {mae:>12,.3f} BPD")
print(f"   R²   : {r2:>12.4f}")
print("=" * 45)

# ─────────────────────────────────────────────
# 7. Save model ONLY (no scaler needed)
# ─────────────────────────────────────────────
os.makedirs("models/extraction", exist_ok=True)
joblib.dump(model, "models/extraction/extraction_model.pkl")
print("\n💾 Model saved → models/extraction/extraction_model.pkl")
print("   ✅ No scaler saved — raw inputs work directly.\n")

# ─────────────────────────────────────────────
# 8. Quick sanity check — prediction range
# ─────────────────────────────────────────────
print("🔍 Sanity check — prediction spread on test set:")
print(f"   Min predicted : {y_pred.min():>10,.2f} BPD")
print(f"   Max predicted : {y_pred.max():>10,.2f} BPD")
print(f"   Std predicted : {y_pred.std():>10,.2f} BPD")
print(f"   (If all values are identical → data is still scaled)")

ModuleNotFoundError: No module named 'pipelines'

In [15]:
import pandas as pd
df = pd.read_csv(r"E:\Research_Paper\oil_gas_ai\data\features\extraction_training_dataset.csv")
print("MEANS:")
print(df[['pressure_gas_lift','pressure_production_casing','pressure_toptubing',
          'temp_toptubing','temp_downholegauge','well_age_years',
          'attribute_score','aggregate_quality_score',
          'mtbf_days','mttr_days','availability_pct']].mean())
print("\nSTDS:")
print(df[['pressure_gas_lift','pressure_production_casing','pressure_toptubing',
          'temp_toptubing','temp_downholegauge','well_age_years',
          'attribute_score','aggregate_quality_score',
          'mtbf_days','mttr_days','availability_pct']].std())

MEANS:
pressure_gas_lift            -7.105427e-15
pressure_production_casing    0.000000e+00
pressure_toptubing           -3.721932e-16
temp_toptubing               -6.268517e-16
temp_downholegauge            1.421085e-14
well_age_years                1.343254e-16
attribute_score               7.835647e-17
aggregate_quality_score       2.686507e-16
mtbf_days                     3.017074e-17
mttr_days                    -6.121599e-19
availability_pct              2.939592e-15
dtype: float64

STDS:
pressure_gas_lift             0.0
pressure_production_casing    0.0
pressure_toptubing            1.0
temp_toptubing                1.0
temp_downholegauge            0.0
well_age_years                1.0
attribute_score               1.0
aggregate_quality_score       1.0
mtbf_days                     1.0
mttr_days                     1.0
availability_pct              1.0
dtype: float64
